# 并行实验：成对排序 (Pairwise Ranking)

**评估指标说明：**
根据论文公式 (31)，本实验的核心评估指标是 **系数的 RMSE（即 L2 范数距离）**：
$$ \text{RMSE} = \frac{1}{S} \sum_{s=1}^S \sqrt{\sum_{l=1}^p (\hat{\theta}_{l,s} - \theta_l^*)^2} $$
我们在代码中严格遵循了这一指标。此外，我们保留了“成对相关性 (Pairwise Correlation)”作为辅助业务指标。

In [1]:
import os
import json
import time
import numpy as np
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from multiprocessing import Pool
from utils.runner import run_single_ranking


In [ ]:
import os
import json  # 必须导入，用于保存文件
import numpy as np
from joblib import Parallel, delayed
from utils.runner import run_single_ranking, run_single_aft

# ==========================================
# 新增：自定义 JSON 编码器，处理 NumPy 数据类型
# ==========================================
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.generic):
            return obj.item()
        return super(NumpyEncoder, self).default(obj)

# ==========================================
# 参数设定 (Hyperparameters)
# ==========================================
NUM_RUNS = 200       
NUM_WORKERS = 10     
params = {
    'Experiment': 'Pairwise Ranking',
    'm': 10,
    'n': 100,  # ⚠️ U-统计量 O(n^2)，调参建议 n=100
    'p_prime': 5,
    'p': 50,
    'pc': 0.3,
    'T': 40,
    'W_inner': 5,
    'rho': 0.01,
    'ic_type': 'bic',
    'lambda_candidates': np.logspace(-1.2, -0.7, 10),
    'noise_type': 'exp',
    'noise_scale': 0.5,
    "run_proposed": True, "run_local": True, "run_baselines": True
}

# 路径管理与临时文件夹设置
folder = "ranking"
tmp_folder = f"{folder}/tmp_results"  # 新增临时文件夹
os.makedirs(folder, exist_ok=True)
os.makedirs(tmp_folder, exist_ok=True)
filename = f"{folder}/exp_{folder}_{NUM_RUNS}_{params['noise_type']}_p{params['p']}_pc_{str(params['pc']).replace('.', '')}_rho_{str(params['rho']).replace('.', '')}.json"

# 选择执行器
runner_fn = run_single_ranking if "Ranking" in params["Experiment"] else run_single_aft

# ==========================================
# 新增：安全运行与单次保存包装函数
# ==========================================
def safe_run_and_save(i, current_params):
    try:
        res = runner_fn(i, current_params)
        tmp_file = os.path.join(tmp_folder, f"run_result_{i}.json")
        with open(tmp_file, 'w', encoding='utf-8') as f:
            # 单次保存也使用自定义编码器
            json.dump({'run_id': i, 'result': res}, f, ensure_ascii=False, indent=4, cls=NumpyEncoder)
        return res
    except Exception as e:
        print(f"\n[Error] Worker {i} failed: {str(e)}")
        return {"run_id": i, "status": "failed", "error_message": str(e)}

print(f"Starting Parallel Experiments: {NUM_RUNS} runs...")

# 【关键点】加入 verbose 参数
# 修改为调用 safe_run_and_save 进行单次保存，防止中途崩溃
results = Parallel(n_jobs=NUM_WORKERS, verbose=10)(
    delayed(safe_run_and_save)(i, params) for i in range(NUM_RUNS)
)

# 最终汇总与保存数据
try:
    with open(filename, 'w', encoding='utf-8') as f:
        # 增加 cls=NumpyEncoder 参数解决序列化报错
        json.dump({'parameters': params, 'results': results}, f, ensure_ascii=False, indent=4, cls=NumpyEncoder)
    print(f"\nCompleted! Results saved to {filename}")
except Exception as e:
    print(f"\n[Error] Final save failed: {e}")
    print(f"200次运行的独立结果已经安全保存在了 {tmp_folder} 文件夹中。")

Starting Parallel Experiments: 200 runs...


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   5 tasks      | elapsed: 12.6min
